In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    regexp_replace, col, to_timestamp, split, lit, when
)
import re

# Criar sessão Spark
spark = SparkSession.builder \
    .appName("Processamento INMET") \
    .getOrCreate()

# Caminhos de origem e destino
CAMINHO_BRONZE = "../data/bronze/INMET/*/*.CSV"  # ajuste conforme sua estrutura real
DESTINO_PARQUET = "../data/bronze/INMET_PARQUET"

# Função para limpar strings estilo cabeçalho
def normalizar_cabecalho(valor):
    valor = re.sub(r'[^a-zA-Z0-9 ]', '', valor.upper())
    valor = valor.replace("REGIO", "REGIAO").replace("ESTACO", "ESTACAO")
    return valor

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/02 23:58:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Leitura bruta dos arquivos CSV
df = spark.read.csv(
    CAMINHO_BRONZE,
    sep=";",
    header=True,
    encoding="latin1"
)

# Renomear colunas variáveis conforme padrão encontrado
if "DATA (YYYY-MM-DD)" in df.columns:
    df = df.select(
        col("DATA (YYYY-MM-DD)").alias("Data"),
        col("TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)").alias("Temperatura")
    )
else:
    df = df.select(
        col("Data"),
        col("TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)").alias("Temperatura")
    )

# Converter Temperatura para float e filtrar -9999
df = df.withColumn("Temperatura", regexp_replace("Temperatura", ",", ".").cast("double"))
df = df.filter(col("Temperatura") != -9999)

# Converter Data e extrair hora
df = df.withColumn("Data", to_timestamp("Data"))
df = df.withColumn("Tempo", split(col("Data").cast("string"), " ")[1])

# Extração de ano com base no caminho do arquivo (_metadata disponível no Spark 3.3+)
df = df.withColumn("caminho", col("_metadata.file_path"))
df = df.withColumn("Ano", regexp_extract(col("caminho"), r"/(\d{4})/", 1))

# Placeholder para Regiao, UF, Estacao, Latitude, Longitude, Altitude
# OBS: Spark não permite leitura parcial de cabeçalho facilmente. Se quiser manter a lógica exata,
# o cabeçalho deve ser lido separadamente com sc.textFile por arquivo.
df = df.withColumn("Regiao", lit("DESCONHECIDO"))
df = df.withColumn("UF", lit("DESCONHECIDO"))
df = df.withColumn("Estacao", lit("DESCONHECIDO"))
df = df.withColumn("Latitude", lit("0"))
df = df.withColumn("Longitude", lit("0"))
df = df.withColumn("Altitude", lit("0"))

# Salvar particionado por ano
df.write.mode("overwrite").partitionBy("Ano").parquet(DESTINO_PARQUET)

print("✅ Processamento finalizado com sucesso!")


25/10/03 00:03:23 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/bronze/INMET/*/*.CSV.
java.io.FileNotFoundException: File ../data/bronze/INMET/*/*.CSV does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.anal

KeyboardInterrupt: 

In [4]:
import os

def testar_acesso_pasta(caminho):
    print(f"\n📂 Testando acesso à pasta: {caminho}")

    # Verifica se a pasta existe
    if not os.path.exists(caminho):
        print("❌ A pasta não existe.")
        return

    # Verifica permissões básicas
    print(f"✔ Existe: {os.path.exists(caminho)}")
    print(f"✔ Leitura permitida: {os.access(caminho, os.R_OK)}")
    print(f"✔ Escrita permitida: {os.access(caminho, os.W_OK)}")

    # Tenta listar arquivos
    try:
        arquivos = os.listdir(caminho)
        if arquivos:
            print("📁 Arquivos encontrados:")
            for item in arquivos:
                print("  -", item)
        else:
            print("⚠ A pasta está vazia.")
    except PermissionError:
        print("❌ Sem permissão para listar arquivos.")

# ======================
# EXEMPLO DE USO
# ======================
if __name__ == "__main__":
    testar_acesso_pasta("/opt/spark/work-dir/jobs/data/bronze/INMET/")       
    testar_acesso_pasta("/opt/spark/work-dir/data/bronze/INMET/")           # Linux
    testar_acesso_pasta("../data/bronze/INMET/")    # Caminho relativo



📂 Testando acesso à pasta: /opt/spark/work-dir/jobs/data/bronze/INMET/
❌ A pasta não existe.

📂 Testando acesso à pasta: /opt/spark/work-dir/data/bronze/INMET/
✔ Existe: True
✔ Leitura permitida: True
✔ Escrita permitida: True
📁 Arquivos encontrados:
  - 2000
  - 2001
  - 2002
  - 2003
  - 2004
  - 2005
  - 2006
  - 2007
  - 2008
  - 2009
  - 2010
  - 2011
  - 2012
  - 2013
  - 2014
  - 2015
  - 2016
  - 2017
  - 2018
  - 2019
  - 2020
  - 2021
  - 2022
  - 2023
  - 2024
  - 2025

📂 Testando acesso à pasta: ../data/bronze/INMET/
✔ Existe: True
✔ Leitura permitida: True
✔ Escrita permitida: True
📁 Arquivos encontrados:
  - 2000
  - 2001
  - 2002
  - 2003
  - 2004
  - 2005
  - 2006
  - 2007
  - 2008
  - 2009
  - 2010
  - 2011
  - 2012
  - 2013
  - 2014
  - 2015
  - 2016
  - 2017
  - 2018
  - 2019
  - 2020
  - 2021
  - 2022
  - 2023
  - 2024
  - 2025
